In [ ]:
import re
import os
import cohere
import faiss
import numpy as np
import pandas as pd
from rank_bm25 import BM25Okapi
from typing import List, Dict, Any, Optional

In [ ]:
COHERE_API_KEY     = os.getenv('COHERE_API_KEY')
COHERE_EMBED_MODEL = "embed-english-v3.0"
EMBED_DIM          = 1024
MIN_SCORE          = 0.30   # corrective filter threshold (cosine)
TOP_K              = 5      # final chunks to return after RRF fusion

co = cohere.Client(COHERE_API_KEY)
print("Cohere client ready")

Cohere client ready


In [ ]:
def _tokenize(text: str) -> List[str]:
    return re.findall(r"(?u)\b\w+\b", text.lower())


def _embed_texts(texts: List[str], input_type: str) -> np.ndarray:
    BATCH = 96
    all_vecs: List[List[float]] = []

    for i in range(0, len(texts), BATCH):
        batch    = texts[i : i + BATCH]
        response = co.embed(
            texts      = batch,
            model      = COHERE_EMBED_MODEL,
            input_type = input_type,
        )
        all_vecs.extend(response.embeddings)

    vecs  = np.array(all_vecs, dtype=np.float32)
    norms = np.linalg.norm(vecs, axis=1, keepdims=True)
    norms = np.where(norms == 0, 1.0, norms)
    return vecs / norms


print("Helpers defined")

Helpers defined


In [ ]:
demo_data = {
    "page": [1, 1, 2, 2, 3, 3, 4, 4, 5],
    "text": [
        # Integration & calculus
        "Integration by parts formula: ∫ u dv = uv − ∫ v du. "
        "Choose u using LIATE: Logarithm, Inverse trig, Algebraic, Trig, Exponential.",
        "The integral of x·sin(x) dx is solved by letting u=x and dv=sin(x)dx, "
        "giving −x·cos(x) + sin(x) + C.",

        # Bayes / probability
        "Bayes theorem: P(A|B) = P(B|A)·P(A) / P(B). "
        "Used when we know the likelihood and want the posterior probability.",
        "Conditional probability example: if P(rain)=0.3 and P(wet|rain)=0.9, "
        "then P(rain|wet) = 0.9·0.3 / P(wet).",

        # Differentiation
        "Chain rule: d/dx[f(g(x))] = f′(g(x))·g′(x). "
        "Apply when differentiating composite functions.",
        "Product rule: d/dx[u·v] = u′v + uv′. "
        "Used when two functions are multiplied together.",

        # Matrices
        "A matrix is invertible if and only if its determinant is non-zero. "
        "det([[a,b],[c,d]]) = ad − bc.",
        "Eigenvalues λ satisfy det(A − λI) = 0. "
        "Eigenvectors are non-zero vectors v where Av = λv.",

        # Limits
        "L'Hôpital's rule: if lim f/g = 0/0 or ∞/∞, "
        "then lim f/g = lim f′/g′ provided the latter exists.",
    ]
}

df = pd.DataFrame(demo_data)



In [6]:
df

,page,text
0,1,Integration by parts formula: ∫ u dv = uv − ∫ ...
1,1,The integral of x·sin(x) dx is solved by letti...
2,2,Bayes theorem: P(A|B) = P(B|A)·P(A) / P(B). Us...
3,2,Conditional probability example: if P(rain)=0....
4,3,Chain rule: d/dx[f(g(x))] = f′(g(x))·g′(x). Ap...
5,3,Product rule: d/dx[u·v] = u′v + uv′. Used when...
6,4,A matrix is invertible if and only if its dete...
7,4,Eigenvalues λ satisfy det(A − λI) = 0. Eigenve...
8,5,"L'Hôpital's rule: if lim f/g = 0/0 or ∞/∞, the..."


In [ ]:
chunks: List[str] = []
metadata: List[Dict[str, Any]] = []

for _, row in df.iterrows():
    text = str(row["text"]).strip()
    if not text:
        continue
    chunks.append(text)
    metadata.append({"page": int(row.get("page", 0))})

print(f"Total chunks ready for indexing: {len(chunks)}")
for i, c in enumerate(chunks):
    print(f"\n[{i}] (page {metadata[i]['page']}) {c[:80]}…")

Total chunks ready for indexing: 9

[0] (page 1) Integration by parts formula: ∫ u dv = uv − ∫ v du. Choose u using LIATE: Logari…

[1] (page 1) The integral of x·sin(x) dx is solved by letting u=x and dv=sin(x)dx, giving −x·…

[2] (page 2) Bayes theorem: P(A|B) = P(B|A)·P(A) / P(B). Used when we know the likelihood and…

[3] (page 2) Conditional probability example: if P(rain)=0.3 and P(wet|rain)=0.9, then P(rain…

[4] (page 3) Chain rule: d/dx[f(g(x))] = f′(g(x))·g′(x). Apply when differentiating composite…

[5] (page 3) Product rule: d/dx[u·v] = u′v + uv′. Used when two functions are multiplied toge…

[6] (page 4) A matrix is invertible if and only if its determinant is non-zero. det([[a,b],[c…

[7] (page 4) Eigenvalues λ satisfy det(A − λI) = 0. Eigenvectors are non-zero vectors v where…

[8] (page 5) L'Hôpital's rule: if lim f/g = 0/0 or ∞/∞, then lim f/g = lim f′/g′ provided the…


In [ ]:
EMBED_INPUT_TYPE_DOC = "search_document"

print("Embedding chunks with Cohere…")
doc_vecs = _embed_texts(chunks, EMBED_INPUT_TYPE_DOC)   # (N, 1024) normalised

index = faiss.IndexFlatIP(EMBED_DIM)
index.add(doc_vecs)
print(f"FAISS index built  → {index.ntotal} vectors")

# BM25 sparse index
tokenized_chunks = [_tokenize(c) for c in chunks]
bm25 = BM25Okapi(tokenized_chunks)
print("BM25  index built")

store = {
    "index":            index,
    "chunks":           chunks,
    "metadata":         metadata,
    "bm25":             bm25,
    "tokenized_chunks": tokenized_chunks,
    "doc_vecs":         doc_vecs,
}

Embedding chunks with Cohere…
FAISS index built  → 9 vectors
BM25  index built


In [ ]:

EMBED_INPUT_TYPE_QUERY = "search_query"

def hybrid_crag(query: str, store: dict, top_k: int = TOP_K, min_score: float = MIN_SCORE) -> str:
 
    # ── Step 1: Dense retrieval ───────────────────────────────────────────────
    q_vec        = _embed_texts([query], EMBED_INPUT_TYPE_QUERY)   # (1, 1024)
    _, indices   = store["index"].search(q_vec, 10)
    dense_idx    = indices[0]                                       

    # ── Step 2: Sparse retrieval ──────────────────────────────────────────────
    tokens       = _tokenize(query)
    sparse_scores = store["bm25"].get_scores(tokens)
    sparse_idx   = np.argsort(sparse_scores)[::-1][:10]            # top-10 BM25 positions

    # ── Step 3: Reciprocal Rank Fusion ────────────────────────────────────────
    K   = 60
    rrf: Dict[int, float] = {}

    for rank, idx in enumerate(dense_idx):
        if 0 <= idx < len(store["chunks"]):
            rrf[idx] = rrf.get(idx, 0.0) + 1.0 / (K + rank + 1)

    for rank, idx in enumerate(sparse_idx):
        if 0 <= idx < len(store["chunks"]):
            rrf[idx] = rrf.get(idx, 0.0) + 1.0 / (K + rank + 1)

    fused = sorted(rrf, key=rrf.get, reverse=True)[:top_k]         # top candidates after fusion

    # ── Step 4: Corrective filter ─────────────────────────────────────────────
    results: List[str] = []
    dropped: List[Dict] = []

    for idx in fused:
        if not (0 <= idx < len(store["chunks"])):
            continue
        cos_sim = float(np.dot(q_vec[0], store["doc_vecs"][idx]))
        if cos_sim < min_score:
            dropped.append({"idx": idx, "cos": round(cos_sim, 4), "text": store["chunks"][idx][:60]})
            continue
        page    = store["metadata"][idx].get("page", "?")
        passage = store["chunks"][idx].strip()
        results.append(
            f"[Page {page} | cosine={cos_sim:.3f}]\n{passage}"
        )

    # ── Debug: show what was dropped ──────────────────────────────────────────
    if dropped:
        print(f"  ⚠ Corrective filter dropped {len(dropped)} chunk(s) (cosine < {min_score}):")
        for d in dropped:
            print(f"    idx={d['idx']}  cos={d['cos']}  \"{d['text']}…\"")

    if not results:
        return f"CRAG: No relevant passages found for query: '{query}' (all below cosine threshold {min_score})"

    return (
        f"Hybrid CRAG — {len(results)} passage(s) retrieved:\n\n"
        + "\n\n---\n\n".join(results)
    )


print("hybrid_crag() function ready")

hybrid_crag() function ready


In [ ]:
query_1 = "integration by parts formula"

print("=" * 60)
print(f"QUERY 1: {query_1!r}")
print("=" * 60)

result_1 = hybrid_crag(query_1, store)
print(result_1)

QUERY 1: 'integration by parts formula'
  ⚠ Corrective filter dropped 1 chunk(s) (cosine < 0.3):
    idx=8  cos=0.2755  "L'Hôpital's rule: if lim f/g = 0/0 or ∞/∞, then lim f/g = li…"
Hybrid CRAG — 4 passage(s) retrieved:

[Page 1 | cosine=0.637]
Integration by parts formula: ∫ u dv = uv − ∫ v du. Choose u using LIATE: Logarithm, Inverse trig, Algebraic, Trig, Exponential.

---

[Page 1 | cosine=0.414]
The integral of x·sin(x) dx is solved by letting u=x and dv=sin(x)dx, giving −x·cos(x) + sin(x) + C.

---

[Page 3 | cosine=0.435]
Product rule: d/dx[u·v] = u′v + uv′. Used when two functions are multiplied together.

---

[Page 3 | cosine=0.429]
Chain rule: d/dx[f(g(x))] = f′(g(x))·g′(x). Apply when differentiating composite functions.


In [ ]:
query_2 = "convolutional neural network architecture deep learning"

print("=" * 60)
print(f"QUERY 2: {query_2!r}")
print("=" * 60)

result_2 = hybrid_crag(query_2, store)
print(result_2)

QUERY 2: 'convolutional neural network architecture deep learning'
  ⚠ Corrective filter dropped 5 chunk(s) (cosine < 0.3):
    idx=5  cos=0.1984  "Product rule: d/dx[u·v] = u′v + uv′. Used when two functions…"
    idx=4  cos=0.1837  "Chain rule: d/dx[f(g(x))] = f′(g(x))·g′(x). Apply when diffe…"
    idx=8  cos=0.062  "L'Hôpital's rule: if lim f/g = 0/0 or ∞/∞, then lim f/g = li…"
    idx=7  cos=0.1031  "Eigenvalues λ satisfy det(A − λI) = 0. Eigenvectors are non-…"
    idx=2  cos=0.173  "Bayes theorem: P(A|B) = P(B|A)·P(A) / P(B). Used when we kno…"
CRAG: No relevant passages found for query: 'convolutional neural network architecture deep learning' (all below cosine threshold 0.3)


In [ ]:
def score_breakdown(query: str, store: dict, label: str = ""):
    print(f"\n{'='*60}")
    print(f"SCORE BREAKDOWN  {'— ' + label if label else ''}")
    print(f"Query: {query!r}")
    print(f"{'='*60}")

    q_vec         = _embed_texts([query], EMBED_INPUT_TYPE_QUERY)
    _, indices    = store["index"].search(q_vec, len(store["chunks"]))
    dense_idx     = indices[0]
    dense_scores  = {int(idx): float(np.dot(q_vec[0], store["doc_vecs"][idx]))
                     for idx in dense_idx if 0 <= idx < len(store["chunks"])}

    tokens        = _tokenize(query)
    bm25_scores   = store["bm25"].get_scores(tokens)

    K   = 60
    rrf: Dict[int, float] = {}
    for rank, idx in enumerate(dense_idx):
        if 0 <= idx < len(store["chunks"]):
            rrf[idx] = rrf.get(idx, 0.0) + 1.0 / (K + rank + 1)
    for rank, idx in enumerate(np.argsort(bm25_scores)[::-1]):
        rrf[int(idx)] = rrf.get(int(idx), 0.0) + 1.0 / (K + rank + 1)

    print(f"\n{'IDX':<5} {'COSINE':<10} {'BM25':<10} {'RRF':<10}  CHUNK (first 60 chars)")
    print("-" * 75)
    for idx in sorted(rrf, key=rrf.get, reverse=True)[:8]:
        cos  = dense_scores.get(idx, 0.0)
        bm   = float(bm25_scores[idx])
        r    = rrf[idx]
        flag = " ✓" if cos >= MIN_SCORE else f" ✗ (below {MIN_SCORE})"
        print(f"{idx:<5} {cos:<10.4f} {bm:<10.4f} {r:<10.5f}  {store['chunks'][idx][:60]}…{flag}")

score_breakdown(query_1, store, label="integration by parts")
score_breakdown(query_2, store, label="neural networks (off-topic)")


SCORE BREAKDOWN  — integration by parts
Query: 'integration by parts formula'

IDX   COSINE     BM25       RRF         CHUNK (first 60 chars)
---------------------------------------------------------------------------
0     0.6374     6.4786     0.03279     Integration by parts formula: ∫ u dv = uv − ∫ v du. Choose u… ✓
1     0.4137     0.9930     0.03175     The integral of x·sin(x) dx is solved by letting u=x and dv=… ✓
5     0.4350     0.0000     0.03105     Product rule: d/dx[u·v] = u′v + uv′. Used when two functions… ✓
4     0.4293     0.0000     0.03102     Chain rule: d/dx[f(g(x))] = f′(g(x))·g′(x). Apply when diffe… ✓
8     0.2755     0.0000     0.03102     L'Hôpital's rule: if lim f/g = 0/0 or ∞/∞, then lim f/g = li… ✗ (below 0.3)
7     0.2187     0.0000     0.03033     Eigenvalues λ satisfy det(A − λI) = 0. Eigenvectors are non-… ✗ (below 0.3)
2     0.3574     0.0000     0.03009     Bayes theorem: P(A|B) = P(B|A)·P(A) / P(B). Used when we kno… ✓
6     0.2094     0.0000     0

In [ ]:
print("Effect of changing MIN_SCORE on Query 2 (off-topic):\n")
for threshold in [0.10, 0.20, 0.30, 0.40]:
    result = hybrid_crag(query_2, store, min_score=threshold)
    first_line = result.split("\n")[0]
    print(f"  min_score={threshold}  →  {first_line}")

print("\nEffect of changing MIN_SCORE on Query 1 (on-topic):\n")
for threshold in [0.10, 0.20, 0.30, 0.40]:
    result = hybrid_crag(query_1, store, min_score=threshold)
    first_line = result.split("\n")[0]
    print(f"  min_score={threshold}  →  {first_line}")

Effect of changing MIN_SCORE on Query 2 (off-topic):

  ⚠ Corrective filter dropped 1 chunk(s) (cosine < 0.1):
    idx=8  cos=0.062  "L'Hôpital's rule: if lim f/g = 0/0 or ∞/∞, then lim f/g = li…"
  min_score=0.1  →  Hybrid CRAG — 4 passage(s) retrieved:
  ⚠ Corrective filter dropped 5 chunk(s) (cosine < 0.2):
    idx=5  cos=0.1987  "Product rule: d/dx[u·v] = u′v + uv′. Used when two functions…"
    idx=4  cos=0.1842  "Chain rule: d/dx[f(g(x))] = f′(g(x))·g′(x). Apply when diffe…"
    idx=8  cos=0.062  "L'Hôpital's rule: if lim f/g = 0/0 or ∞/∞, then lim f/g = li…"
    idx=7  cos=0.1032  "Eigenvalues λ satisfy det(A − λI) = 0. Eigenvectors are non-…"
    idx=2  cos=0.1726  "Bayes theorem: P(A|B) = P(B|A)·P(A) / P(B). Used when we kno…"
  min_score=0.2  →  CRAG: No relevant passages found for query: 'convolutional neural network architecture deep learning' (all below cosine threshold 0.2)
  ⚠ Corrective filter dropped 5 chunk(s) (cosine < 0.3):
    idx=5  cos=0.1987  "Product rule: d/dx